In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, BatchNormalization, ReLU, MaxPooling2D, Conv2DTranspose, Input
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Load MNIST dataset
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

# Normalize the dataset
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Reshape to add channel dimension
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print(f"Training data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Training data shape: (60000, 28, 28, 1)
Test data shape: (10000, 28, 28, 1)


In [ ]:
class AutoEncoder:

  """
  This class will be used for defining the Auto encoder functionality
  performs the required operations like convolution, Encoding & Decoding.
  """
  def __init__(self, auto_encoder="sparse", sparsity_level=0.05, lambda_sparse=1e-3, lambda_contractive=1e-4):
    self.auto_encoder = auto_encoder
    self.sparsity_level = sparsity_level
    self.lambda_sparse = lambda_sparse
    self.lambda_contractive = lambda_contractive
    self.encoder_model = None
    self.decoder_model = None


  def convolution(self, image_input, num_filters):

    first_conv = Conv2D(num_filters, kernel_size = (3,3), padding = "same")(image_input)
    batch_norm1 = BatchNormalization()(first_conv)
    output_1 = ReLU()(batch_norm1)

    # output of 1st convolution passed for another convolution
    second_conv = Conv2D(num_filters, kernel_size = (3,3), padding = "same")(output_1)
    batch_norm2 = BatchNormalization()(second_conv)
    output_2 = ReLU()(batch_norm2)

    return output_2


  def encoder(self, input, num_filters):
    """creating encoding layer by downsampling using max pooling and as we don't
    need skip connection then it'll only return downsampled output
    """
    encoded = self.convolution(input, num_filters)
    max_pool = MaxPooling2D(strides = (2,2))(encoded)
    return max_pool


  def decoder(self, input, num_filters):
    # upsampling the encoded data without concatinating skip connections
    upsampled = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(input)
    output = self.convolution(upsampled, num_filters)
    return output

  def contractive_penalty(self, hidden_layer_output, input_layer):
    
    gradients = tf.gradients(hidden_layer_output, input_layer)[0]
    
    # Frobenius norm of the Jacobian matrix
    contractive_loss = tf.reduce_sum(tf.square(gradients), axis=[1, 2, 3])
    contractive_loss = tf.reduce_mean(contractive_loss)
    
    return self.lambda_contractive * contractive_loss

  def sparse_penalty(self, hidden_layer_output):
    y = tf.reduce_mean(hidden_layer_output, axis=0)

    # used KL divergence for penalty
    kl_diverg = tf.reduce_sum(
            self.sparsity_level * tf.math.log(self.sparsity_level / (y + 1e-10)) +
            (1 - self.sparsity_level) * tf.math.log((1 - self.sparsity_level) / (1 - y + 1e-10)))

    return self.lambda_sparse * kl_divergence



  def construct_model(self, input_dimensions):
    input = Input(input_dimensions)
    p1 = self.encoder(input, 8)
    p2 = self.encoder(p1, 16)
    p3 = self.encoder(p2, 32)

    bottle_neck = self.convolution(p3, 64)

    d1 = self.decoder(bottle_neck, 32)
    d2 = self.decoder(d1, 16)
    d3 = self.decoder(d2, 8)

    output = Conv2D(1, 1, padding="same", activation="sigmoid")(d3)

    model = Model(input, output, name="U-Net")
    return model

In [ ]:
class AutoEncoder:
    
    def __init__(self, auto_encoder="sparse", sparsity_level=0.05, lambda_sparse=1e-3, lambda_contractive=1e-4):
        self.auto_encoder = auto_encoder
        self.sparsity_level = sparsity_level
        self.lambda_sparse = lambda_sparse
        self.lambda_contractive = lambda_contractive
        self.encoder_model = None
        self.decoder_model = None

    def convolution(self, image_input, num_filters):
        
        first_conv = Conv2D(num_filters, kernel_size=(3,3), padding="same")(image_input)
        batch_norm1 = BatchNormalization()(first_conv)
        output_1 = ReLU()(batch_norm1)

        second_conv = Conv2D(num_filters, kernel_size=(3,3), padding="same")(output_1)
        batch_norm2 = BatchNormalization()(second_conv)
        output_2 = ReLU()(batch_norm2)

        return output_2

    def encoder(self, input_layer, num_filters):
        
        encoded = self.convolution(input_layer, num_filters)
        max_pool = MaxPooling2D(strides=(2,2))(encoded)
        return max_pool

    def decoder(self, input_layer, num_filters):
        
        upsampled = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(input_layer)
        output = self.convolution(upsampled, num_filters)
        return output

    def sparse_penalty(self, hidden_layer_output):
      
        mean_activation = tf.reduce_mean(hidden_layer_output, axis=0)
        
        kl_divergence = tf.reduce_sum(
            self.sparsity_level * tf.math.log(self.sparsity_level / (mean_activation + 1e-10)) +
            (1 - self.sparsity_level) * tf.math.log((1 - self.sparsity_level) / (1 - mean_activation + 1e-10))
        )
        
        return self.lambda_sparse * kl_divergence

    def contractive_penalty(self, hidden_layer_output, input_layer):
      
        gradients = tf.gradients(hidden_layer_output, input_layer)[0]
        
        contractive_loss = tf.reduce_sum(tf.square(gradients), axis=[1, 2, 3])
        contractive_loss = tf.reduce_mean(contractive_loss)
        
        return self.lambda_contractive * contractive_loss

    def construct_model(self, input_dimensions):
    
        input_layer = Input(input_dimensions)
        
        # Encoder path - carefully designed for 28x28 input
        p1 = self.encoder(input_layer, 16)   # 28x28 -> 14x14
        p2 = self.encoder(p1, 32)            # 14x14 -> 7x7
        
        # Bottleneck - deepest point
        bottleneck = self.convolution(p2, 64)  # 7x7 with 64 filters
        
        # Decoder path - mirror the encoder
        d1 = self.decoder(bottleneck, 32)    # 7x7 -> 14x14
        d2 = self.decoder(d1, 16)            # 14x14 -> 28x28
        
        # Output layer
        output = Conv2D(1, 1, padding="same", activation="sigmoid")(d2)
        
        # Create full autoencoder model
        autoencoder = Model(input_layer, output, name=f"{self.auto_encoder}_autoencoder")
        
        # Create separate encoder model for embedding extraction
        self.encoder_model = Model(input_layer, bottleneck, name="encoder")
        
        # Create separate decoder model - fix the input shape issue
        decoder_input = Input(shape=(7, 7, 64))  # Explicit shape for 7x7x64 bottleneck
        decoder_d1 = self.decoder(decoder_input, 32)
        decoder_d2 = self.decoder(decoder_d1, 16)
        decoder_output = Conv2D(1, 1, padding="same", activation="sigmoid")(decoder_d2)
        self.decoder_model = Model(decoder_input, decoder_output, name="decoder")
        
        return autoencoder, self.encoder_model, self.decoder_model

    def custom_loss(self, y_true, y_pred):
        """Custom loss function that includes regularization terms"""
        # Base reconstruction loss (MSE)
        mse_loss = tf.reduce_mean(tf.square(y_true - y_pred))
        
        if self.auto_encoder == "sparse":
            # Get hidden layer output (bottleneck)
            hidden_output = self.encoder_model(y_true)
            sparse_loss = self.sparse_penalty(hidden_output)
            return mse_loss + sparse_loss
            
        elif self.auto_encoder == "contractive":
            # Get hidden layer output and calculate contractive penalty
            hidden_output = self.encoder_model(y_true)
            contractive_loss = self.contractive_penalty(hidden_output, y_true)
            return mse_loss + contractive_loss
            
        else:
            return mse_loss

In [7]:
# Implement Sparse Autoencoder
print("=== Training Sparse Autoencoder ===")

sparse_ae = AutoEncoder(auto_encoder="sparse", sparsity_level=0.05, lambda_sparse=1e-3)
sparse_model, sparse_encoder, sparse_decoder = sparse_ae.construct_model((28, 28, 1))

# Compile with custom loss
sparse_model.compile(
    optimizer='adam',
    loss=sparse_ae.custom_loss,
    metrics=['mse']
)

# Display model architecture
sparse_model.summary()

# Train the sparse autoencoder
print("\nTraining Sparse Autoencoder...")
sparse_history = sparse_model.fit(
    x_train, x_train,
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test, x_test),
    verbose=1
)

=== Training Sparse Autoencoder ===


Model: "sparse_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 28, 28, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 28, 28, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_20 (ReLU)                 │ (None, 28, 28, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 28, 28, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 28, 28, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_21 (ReLU)                 │ (None, 28, 28, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 14, 14, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 14, 14, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_22 (ReLU)                 │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_25 (Conv2D)              │ (None, 14, 14, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (None, 14, 14, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_23 (ReLU)                 │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 7, 7, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 7, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_24 (ReLU)                 │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 7, 7, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 7, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_25 (ReLU)                 │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_6              │ (None, 14, 14, 32)     │         8,22

 Total params: 106,513 (416.07 KB)

 Trainable params: 105,873 (413.57 KB)

 Non-trainable params: 640 (2.50 KB)


Training Sparse Autoencoder...
Epoch 1/20
Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 138s 272ms/step - loss: nan - mse: 0.0475 - val_loss: nan - val_mse: 0.1139
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 138s 272ms/step - loss: nan - mse: 0.0475 - val_loss: nan - val_mse: 0.1139
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 121s 258ms/step - loss: 0.0346 - mse: 0.0118 - val_loss: 0.1260 - val_mse: 0.0737
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 121s 258ms/step - loss: 0.0346 - mse: 0.0118 - val_loss: 0.1260 - val_mse: 0.0737
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 177s 378ms/step - loss: 0.0197 - mse: 0.0080 - val_loss: 0.0918 - val_mse: 0.0519
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 177s 378ms/step - loss: 0.0197 - mse: 0.0080 - val_loss: 0.0918 - val_mse: 0.0519
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 138s 294ms/step - loss: 0.0154 - mse: 0.0064 - val_loss: 0.0863 - val_mse: 0.0479
Epoch 5/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 138s 294ms/step - loss: 0.0154 - mse: 0.0064 - val_loss: 0.0863 - val_mse: 

In [ ]:
# Implement Contractive Autoencoder
print("\n=== Training Contractive Autoencoder ===")

contractive_ae = AutoEncoder(auto_encoder="contractive", lambda_contractive=1e-4)
contractive_model, contractive_encoder, contractive_decoder = contractive_ae.construct_model((28, 28, 1))

# Compile with custom loss
contractive_model.compile(
    optimizer='adam',
    loss=contractive_ae.custom_loss,
    metrics=['mse']
)

# Display model architecture
contractive_model.summary()

# Train the contractive autoencoder
print("\nTraining Contractive Autoencoder...")
contractive_history = contractive_model.fit(
    x_train, x_train,
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test, x_test),
    verbose=1
)

In [ ]:
# Visualization and Comparison
def plot_training_history(sparse_hist, contractive_hist):
    """Plot training history for both models"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Sparse Autoencoder Loss
    axes[0, 0].plot(sparse_hist.history['loss'], label='Training Loss', color='blue')
    axes[0, 0].plot(sparse_hist.history['val_loss'], label='Validation Loss', color='red')
    axes[0, 0].set_title('Sparse Autoencoder - Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Sparse Autoencoder MSE
    axes[0, 1].plot(sparse_hist.history['mse'], label='Training MSE', color='blue')
    axes[0, 1].plot(sparse_hist.history['val_mse'], label='Validation MSE', color='red')
    axes[0, 1].set_title('Sparse Autoencoder - MSE')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('MSE')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Contractive Autoencoder Loss
    axes[1, 0].plot(contractive_hist.history['loss'], label='Training Loss', color='green')
    axes[1, 0].plot(contractive_hist.history['val_loss'], label='Validation Loss', color='orange')
    axes[1, 0].set_title('Contractive Autoencoder - Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Contractive Autoencoder MSE
    axes[1, 1].plot(contractive_hist.history['mse'], label='Training MSE', color='green')
    axes[1, 1].plot(contractive_hist.history['val_mse'], label='Validation MSE', color='orange')
    axes[1, 1].set_title('Contractive Autoencoder - MSE')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('MSE')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()

plot_training_history(sparse_history, contractive_history)

In [ ]:
def visualize_reconstructions(models, encoders, decoders, test_data, n_samples=10):
    """Visualize original vs reconstructed images for both models"""
    
    sparse_model, contractive_model = models
    sparse_encoder, contractive_encoder = encoders
    sparse_decoder, contractive_decoder = decoders
    
    # Select random test samples
    indices = np.random.choice(len(test_data), n_samples, replace=False)
    test_samples = test_data[indices]
    
    # Get reconstructions
    sparse_reconstructed = sparse_model.predict(test_samples)
    contractive_reconstructed = contractive_model.predict(test_samples)
    
    # Get embeddings h = E(I)
    sparse_embeddings = sparse_encoder.predict(test_samples)
    contractive_embeddings = contractive_encoder.predict(test_samples)
    
    # Verify decoder outputs Î = D(h)
    sparse_decoded = sparse_decoder.predict(sparse_embeddings)
    contractive_decoded = contractive_decoder.predict(contractive_embeddings)
    
    # Plot results
    fig, axes = plt.subplots(5, n_samples, figsize=(n_samples*2, 10))
    
    for i in range(n_samples):
        # Original images
        axes[0, i].imshow(test_samples[i].squeeze(), cmap='gray')
        axes[0, i].set_title(f'Original {i+1}')
        axes[0, i].axis('off')
        
        # Sparse reconstructions
        axes[1, i].imshow(sparse_reconstructed[i].squeeze(), cmap='gray')
        axes[1, i].set_title(f'Sparse Recon')
        axes[1, i].axis('off')
        
        # Contractive reconstructions
        axes[2, i].imshow(contractive_reconstructed[i].squeeze(), cmap='gray')
        axes[2, i].set_title(f'Contractive Recon')
        axes[2, i].axis('off')
        
        # Sparse decoder output D(E(I))
        axes[3, i].imshow(sparse_decoded[i].squeeze(), cmap='gray')
        axes[3, i].set_title(f'Sparse D(E(I))')
        axes[3, i].axis('off')
        
        # Contractive decoder output D(E(I))
        axes[4, i].imshow(contractive_decoded[i].squeeze(), cmap='gray')
        axes[4, i].set_title(f'Contractive D(E(I))')
        axes[4, i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return sparse_embeddings, contractive_embeddings

# Visualize results
models = [sparse_model, contractive_model]
encoders = [sparse_encoder, contractive_encoder]
decoders = [sparse_decoder, contractive_decoder]

sparse_emb, contractive_emb = visualize_reconstructions(models, encoders, decoders, x_test)

In [ ]:
def analyze_embeddings(sparse_embeddings, contractive_embeddings):
    """Analyze the learned embeddings h = E(I)"""
    
    print("=== Embedding Analysis ===")
    print(f"Sparse embedding shape: {sparse_embeddings.shape}")
    print(f"Contractive embedding shape: {contractive_embeddings.shape}")
    
    # Calculate sparsity of embeddings
    sparse_sparsity = np.mean(sparse_embeddings < 0.1)  # Low activation threshold
    contractive_sparsity = np.mean(contractive_embeddings < 0.1)
    
    print(f"\nSparsity Analysis (values < 0.1):")
    print(f"Sparse autoencoder embedding sparsity: {sparse_sparsity:.3f}")
    print(f"Contractive autoencoder embedding sparsity: {contractive_sparsity:.3f}")
    
    # Calculate embedding statistics
    print(f"\nEmbedding Statistics:")
    print(f"Sparse - Mean: {np.mean(sparse_embeddings):.4f}, Std: {np.std(sparse_embeddings):.4f}")
    print(f"Contractive - Mean: {np.mean(contractive_embeddings):.4f}, Std: {np.std(contractive_embeddings):.4f}")
    
    # Visualize embedding distributions
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(sparse_embeddings.flatten(), bins=50, alpha=0.7, color='blue')
    axes[0].set_title('Sparse Autoencoder Embedding Distribution')
    axes[0].set_xlabel('Activation Value')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(contractive_embeddings.flatten(), bins=50, alpha=0.7, color='green')
    axes[1].set_title('Contractive Autoencoder Embedding Distribution')
    axes[1].set_xlabel('Activation Value')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def calculate_reconstruction_error(models, test_data):
    """Calculate reconstruction errors for both models"""
    sparse_model, contractive_model = models
    
    sparse_recon = sparse_model.predict(test_data)
    contractive_recon = contractive_model.predict(test_data)
    
    sparse_mse = np.mean(np.square(test_data - sparse_recon))
    contractive_mse = np.mean(np.square(test_data - contractive_recon))
    
    print(f"\n=== Reconstruction Error Analysis ===")
    print(f"Sparse Autoencoder MSE: {sparse_mse:.6f}")
    print(f"Contractive Autoencoder MSE: {contractive_mse:.6f}")
    
    return sparse_mse, contractive_mse

# Analyze embeddings and reconstruction errors
analyze_embeddings(sparse_emb, contractive_emb)
sparse_mse, contractive_mse = calculate_reconstruction_error(models, x_test[:1000])

In [ ]:
# Demonstrate E(I) and D(h) functionality
def demonstrate_encoder_decoder_pipeline(test_image, sparse_encoder, sparse_decoder, 
                                       contractive_encoder, contractive_decoder):
    """
    Demonstrate the complete pipeline: I -> E(I) = h -> D(h) = Î
    """
    print("=== Encoder-Decoder Pipeline Demonstration ===")
    
    # Single test image
    I = test_image.reshape(1, 28, 28, 1)
    
    # Sparse Autoencoder Pipeline
    h_sparse = sparse_encoder.predict(I)  # h = E(I)
    I_hat_sparse = sparse_decoder.predict(h_sparse)  # Î = D(h)
    
    # Contractive Autoencoder Pipeline
    h_contractive = contractive_encoder.predict(I)  # h = E(I)
    I_hat_contractive = contractive_decoder.predict(h_contractive)  # Î = D(h)
    
    print(f"Original image shape: {I.shape}")
    print(f"Sparse embedding h shape: {h_sparse.shape}")
    print(f"Sparse reconstructed Î shape: {I_hat_sparse.shape}")
    print(f"Contractive embedding h shape: {h_contractive.shape}")
    print(f"Contractive reconstructed Î shape: {I_hat_contractive.shape}")
    
    # Calculate compression ratio
    original_size = np.prod(I.shape[1:])
    embedding_size = np.prod(h_sparse.shape[1:])
    compression_ratio = original_size / embedding_size
    
    print(f"\nCompression Analysis:")
    print(f"Original size: {original_size} pixels")
    print(f"Embedding size: {embedding_size} features")
    print(f"Compression ratio: {compression_ratio:.1f}x")
    
    # Visualize the pipeline
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    
    # Sparse pipeline
    axes[0, 0].imshow(I.squeeze(), cmap='gray')
    axes[0, 0].set_title('Original Image I')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(h_sparse.squeeze(), cmap='viridis', aspect='auto')
    axes[0, 1].set_title('Sparse Embedding h = E(I)')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(I_hat_sparse.squeeze(), cmap='gray')
    axes[0, 2].set_title('Sparse Reconstruction Î = D(h)')
    axes[0, 2].axis('off')
    
    # Contractive pipeline
    axes[1, 0].imshow(I.squeeze(), cmap='gray')
    axes[1, 0].set_title('Original Image I')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(h_contractive.squeeze(), cmap='viridis', aspect='auto')
    axes[1, 1].set_title('Contractive Embedding h = E(I)')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(I_hat_contractive.squeeze(), cmap='gray')
    axes[1, 2].set_title('Contractive Reconstruction Î = D(h)')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return h_sparse, I_hat_sparse, h_contractive, I_hat_contractive

# Demonstrate with a test image
test_image = x_test[0]
h_s, I_hat_s, h_c, I_hat_c = demonstrate_encoder_decoder_pipeline(
    test_image, sparse_encoder, sparse_decoder, contractive_encoder, contractive_decoder
)

In [ ]:
# Display Interpolation Results in Formatted Table
import pandas as pd

def display_interpolation_table(new_images):
    """
    Create clean formatted tables for PSNR and L2 results
    """
    # Extract data structure
    image_pairs = list(new_images.keys())
    alpha_values = sorted(list(set(
        alpha for pair in new_images.values() 
        for alpha in pair["alpha"].keys()
    )))
    
    # Prepare data for tables
    data = []
    for pair in image_pairs:
        for alpha in alpha_values:
            if alpha in new_images[pair]["alpha"]:
                metrics = new_images[pair]["alpha"][alpha]
                data.append({
                    'Image_Pair': pair,
                    'Alpha': alpha,
                    'PSNR_SAE': f"{metrics['psnr_sae']:.3f}",
                    'PSNR_CAE': f"{metrics['psnr_cae']:.3f}",
                    'L2_SAE': f"{metrics['l2_sae']:.4f}",
                    'L2_CAE': f"{metrics['l2_cae']:.4f}"
                })
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Display main results table
    print("="*80)
    print("INTERPOLATION RESULTS: AUTOENCODER COMPARISON")
    print("="*80)
    print("\n📊 PSNR (Peak Signal-to-Noise Ratio) - Higher is Better")
    print("📏 L2 Norm (Embedding Distance) - Lower is Better")
    print("-"*80)
    
    # Pivot tables for better readability
    psnr_comparison = df.pivot_table(
        index='Image_Pair', 
        columns='Alpha', 
        values=['PSNR_SAE', 'PSNR_CAE'], 
        aggfunc='first'
    )
    
    l2_comparison = df.pivot_table(
        index='Image_Pair', 
        columns='Alpha', 
        values=['L2_SAE', 'L2_CAE'], 
        aggfunc='first'
    )
    
    print("\n🔵 PSNR VALUES:")
    print(psnr_comparison.to_string())
    
    print("\n📏 L2 NORM VALUES:")
    print(l2_comparison.to_string())
    
    # Summary statistics
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    
    psnr_sae_avg = df['PSNR_SAE'].astype(float).mean()
    psnr_cae_avg = df['PSNR_CAE'].astype(float).mean()
    l2_sae_avg = df['L2_SAE'].astype(float).mean()
    l2_cae_avg = df['L2_CAE'].astype(float).mean()
    
    summary = pd.DataFrame({
        'Metric': ['PSNR', 'L2 Norm'],
        'SAE_Avg': [f"{psnr_sae_avg:.3f}", f"{l2_sae_avg:.4f}"],
        'CAE_Avg': [f"{psnr_cae_avg:.3f}", f"{l2_cae_avg:.4f}"],
        'Winner': [
            'CAE' if psnr_cae_avg > psnr_sae_avg else 'SAE',
            'SAE' if l2_sae_avg < l2_cae_avg else 'CAE'
        ]
    })
    
    print(summary.to_string(index=False))
    
    # Final verdict
    psnr_winner = "Contractive" if psnr_cae_avg > psnr_sae_avg else "Sparse"
    l2_winner = "Sparse" if l2_sae_avg < l2_cae_avg else "Contractive"
    
    print(f"\n🏆 RECONSTRUCTION QUALITY: {psnr_winner} Autoencoder")
    print(f"🎯 EMBEDDING INTERPOLATION: {l2_winner} Autoencoder")
    
    if psnr_winner == l2_winner:
        print(f"\n✅ OVERALL WINNER: {psnr_winner} Autoencoder")
    else:
        print(f"\n⚖️ MIXED RESULTS - Context-dependent performance")

# Usage: Call this function after running your interpolation code
# display_interpolation_table(new_images)